# SOXX 2023–2026 评价段原始 N-PORT 来源

固定 14 份首版原始报告和一份 2025-09 迟到修订的 SEC filing index/XML URL，原字节及响应头 NAS 备份。原件身份、数值与时序各自核验；本卡无 SOXX 收益、拟合或账户。

In [1]:
import json,hashlib,subprocess,time,tarfile,tempfile,shlex
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_soxx_2023_2026_nport_original_source_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'attempt_started.json').exists() and sha(root/plan['parent_manifest'])==plan['parent_manifest_sha256']
raw=root/'data/snapshots/stage1_soxx_2023_2026_nport_original_source_20261023';raw.mkdir(parents=True,exist_ok=True);assert not any(raw.iterdir())
state={'status':'started','GET_attempts':0,'index_GETs':0,'primary_XML_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];error=None;specs=[]
for item in plan['exact_filings']:
 for kind,url,limit in [('index',item['index_url'],plan['request']['index_max_bytes_each']),('XML',item['primary_xml_url'],plan['request']['XML_max_bytes_each'])]:specs.append((item,kind,url,limit))
assert len(specs)==30
for i,(item,kind,url,limit) in enumerate(specs):
 state['GET_attempts']+=1;state['index_GETs' if kind=='index' else 'primary_XML_GETs']+=1;(card/'attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
 stem=item['accession']+'.'+kind;body=raw/(stem+('.html' if kind=='index' else '.xml'));headers=raw/(stem+'.headers');started=datetime.now(timezone.utc).isoformat();r=plan['request']
 cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(r['connect_timeout_seconds']),'--max-time',str(r['total_timeout_seconds_each']),'--max-filesize',str(limit),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',r['user_agent'],'-w','%{http_code} %{url_effective}',url]
 try:
  run=subprocess.run(cmd,capture_output=True,text=True,timeout=60);fields=run.stdout.rsplit(' ',1);status=int(fields[0]) if len(fields)==2 and fields[0].isdigit() else None
  receipt={'accession':item['accession'],'kind':kind,'url':url,'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':fields[-1] if fields else None,'stderr':run.stderr[:300]}
 except Exception as exc:receipt={'accession':item['accession'],'kind':kind,'url':url,'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':type(exc).__name__+':'+str(exc)[:180]}
 receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0)
 receipts.append(receipt)
 if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not body.exists() or body.stat().st_size>limit:error='transport_or_size_failure';break
 if i+1<len(specs):time.sleep(r['minimum_seconds_between_requests'])
(card/'capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
manifest={'plan_sha256':sha(card/'plan.json'),'files':[{'name':p.name,'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(raw.iterdir()) if p.is_file()],'capture_receipts_sha256':sha(card/'capture_receipts.json')};(card/'source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='soxx-held-nport-') as temp:
 archive=Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for x in manifest['files']:out.add(raw/x['name'],arcname='raw/'+x['name'])
  out.add(card/'source_manifest.json',arcname='source_manifest.json');out.add(card/'capture_receipts.json',arcname='capture_receipts.json')
 digest=sha(archive);evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='soxx-held-nport-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as source:
   for x in manifest['files']:
    member=source.extractfile('raw/'+x['name']);assert member and hashlib.sha256(member.read()).hexdigest()==x['sha256']
result={'status':'fifteen_held_index_and_XML_pairs_archived_parse_pending' if not error and len(receipts)==30 else 'partial_held_source_transport_needs_repair','error':error,'GET_attempts':state['GET_attempts'],'index_GETs':state['index_GETs'],'primary_XML_GETs':state['primary_XML_GETs'],'HTTP200_files':sum(x.get('http_status')==200 and x.get('curl_exit_code')==0 for x in receipts),'nas_archive_sha256':digest,'nas_archive_bytes':evidence['size_bytes'],'nas_remote_and_scoped_restore_verified':True,'market_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'transport_result.json').write_text(json.dumps(result,indent=2)+'\n')
print(result)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'fifteen_held_index_and_XML_pairs_archived_parse_pending', 'error': None, 'GET_attempts': 30, 'index_GETs': 15, 'primary_XML_GETs': 15, 'HTTP200_files': 30, 'nas_archive_sha256': '650e7aae26f43392f7fe0c44f52ac963ad92b6a3170b1f1c7c084134cca5b455', 'nas_archive_bytes': 173712, 'nas_remote_and_scoped_restore_verified': True, 'market_outcome_reads': 0, 'model_fits': 0, 'native_accounts': 0}


## 中文结论

2023–2026 评价段 14 份首版与 1 份 2025-09 报告迟到修订的 SEC 索引/XML 共 30 个文件全部 HTTP 200，NAS 原字节归档 SHA-256 `650e7aae26f43392f7fe0c44f52ac963ad92b6a3170b1f1c7c084134cca5b455` 远端哈希和限定恢复通过。零网络索引/XML 对读确认 15 份均属 S000004354/C000012084/SOXX，报告期、申报日和每份 30 条 EC 匹配。尚未解析股值及修订差异，0 市场收益、0 拟合、0 账户。